# Faster R-CNN — Forge Sweeps (Kaggle T4 × 2)

Notebook riêng; không thay notebook 03 của nhóm. Dùng lần lượt **preflight → pilot → tuning**.
Mỗi campaign tách riêng, một agent chạy tuần tự, mỗi trial dùng hai GPU.
Objective là **best validation mAP50–95**, không phải test hay last epoch.

Trước chạy: commit/push code đã kiểm; điền SHA 40 ký tự vào `REPO_REF`.
Kaggle: Add Input cache đã tiền xử lý, GPU T4 × 2, Internet On,
Settings → Secrets → bật `WANDB_API_KEY`. Không ghi hoặc in key trong notebook.
Mặc định notebook chỉ chuẩn bị; đăng ký và chạy agent cần bật hai công tắc riêng bên dưới.

In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_REF = ""  # SHA đầy đủ của commit có tích hợp Sweeps, đã push; không dùng tên branch.
assert re.fullmatch(r"[0-9a-f]{40}", REPO_REF), "Điền SHA đã commit/push trước khi chạy."
assert Path("/kaggle/working").exists(), "Notebook này dành cho Kaggle."
REPO = Path("/kaggle/working/VN-Traffic-Sign-Detection")

def run(command, **kwargs):
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty.strip(), "Repo có tracked changes; giữ lại trước khi checkout."
run(["git", "-C", REPO, "fetch", "origin", REPO_REF])
run(["git", "-C", REPO, "checkout", "--detach", REPO_REF])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[detect]", "wandb==0.31.0"])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Kiểm GPU và secret

Cell này chưa tạo run hoặc Sweep. API host lấy từ config chung của nhóm.
Lỗi CUDA/NMS hoặc thiếu GPU: dừng, không tự giảm batch hay số GPU.

In [ ]:
import torch, torchvision, wandb
from torchvision.ops import nms
from kaggle_secrets import UserSecretsClient
from vn_tsd.utils.io import load_yaml, save_yaml

shared = load_yaml(REPO / "configs/shared.yaml")
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_BASE_URL"] = shared["project"]["wandb_base_url"]
os.environ["WANDB_MODE"] = "online"
print("torch", torch.__version__, "torchvision", torchvision.__version__, "wandb", wandb.__version__)
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Cần đúng hai GPU T4."
for index in range(2):
    device = f"cuda:{index}"
    result = nms(torch.tensor([[0., 0., 10., 10.]], device=device), torch.tensor([.9], device=device), .5)
    assert result.tolist() == [0]
    print(index, torch.cuda.get_device_name(index))

## 2. Chọn giai đoạn và chuẩn bị offline

**preflight:** một trial cố định, 2 epoch; chỉ kiểm kỹ thuật, không xếp hạng với full train.
**pilot:** một trial baseline cố định, ngân sách epoch hiện hành trong YAML (hiện 100), patience hiện hành.
Đọc best/last epoch, LR, early stopping, thời gian, VRAM rồi chốt budget trước tuning.
**tuning:** search space LR/WD/dropout/jitter; phải điền `TUNING_RUN_CAP` và `TUNING_EPOCHS` sau pilot.

`TEST_POLICY="shared"` giữ policy auto-test của config nhóm. Nếu nhóm đồng ý không đánh giá test
trong tuning, đổi thành `"skip"` cho một campaign mới. Không dùng test để chọn trial.
Chuẩn bị sẽ quét/hash toàn cache, có thể mất vài phút; không copy dataset vào Working.

In [ ]:
from vn_tsd.config.resolve import resolve_config
from vn_tsd.pipelines.faster_rcnn.sweep import prepare, load_campaign, PARAMETERS, OBJECTIVE

STAGE = "preflight"  # preflight / pilot / tuning
CAMPAIGN_NAME = "frcnn_preflight_s42_v1"  # tên mới mỗi campaign; giữ tên khi tiếp tục cùng campaign
CAMPAIGN = Path("/kaggle/working/frcnn_sweeps") / CAMPAIGN_NAME
DATA_ROOT = Path(load_yaml("configs/runtime/kaggle.yaml")["data_root_default"])
TEST_POLICY = "shared"
TUNING_RUN_CAP = None  # chốt sau pilot, ví dụ hữu hạn; không tự dùng 12 như budget đã duyệt
TUNING_EPOCHS = None   # chốt sau pilot; đổi cap làm đổi cosine từ đầu
assert STAGE in {"preflight", "pilot", "tuning"}
assert DATA_ROOT.is_dir(), f"Kiểm lại Add Input/cache path: {DATA_ROOT}"

overrides = {"data": {"root": str(DATA_ROOT)}, "seed": 42}
if STAGE == "preflight":
    overrides["train"] = {"epochs": 2}
elif STAGE == "tuning":
    assert type(TUNING_RUN_CAP) is int and TUNING_RUN_CAP > 0
    assert type(TUNING_EPOCHS) is int and TUNING_EPOCHS > 0
    overrides["train"] = {"epochs": TUNING_EPOCHS}
cfg = resolve_config("configs/pipelines/faster_rcnn.yaml", "configs/shared.yaml",
                     "configs/runtime/kaggle.yaml", overrides=overrides)
search = load_yaml("configs/sweeps/faster_rcnn.yaml")
cap = TUNING_RUN_CAP if STAGE == "tuning" else 1
if STAGE != "tuning":
    search = {"method": "grid", "metric": {"name": OBJECTIVE, "goal": "maximize"},
              "parameters": {key: {"value": cfg[key.split('.')[0]][key.split('.')[1]]} for key in PARAMETERS}}
if CAMPAIGN.exists():
    frozen, existing, existing_search = load_campaign(CAMPAIGN)
    # Existing campaigns are immutable. Avoid silently resuming the wrong stage/settings.
    from vn_tsd.pipelines.faster_rcnn.dataset import content_hash
    requested = json.loads((CAMPAIGN / "notebook_request.json").read_text())
    assert requested == {"stage": STAGE, "cap": cap, "test_policy": TEST_POLICY, "cfg": cfg, "search": search}
    print("Giữ campaign đã chuẩn bị:", CAMPAIGN)
else:
    prepare(CAMPAIGN, cfg, search, run_cap=cap, nproc=2, test_policy=TEST_POLICY)
    (CAMPAIGN / "notebook_request.json").write_text(json.dumps(
        {"stage": STAGE, "cap": cap, "test_policy": TEST_POLICY, "cfg": cfg, "search": search}, indent=2))
print((CAMPAIGN / "base_config.yaml").read_text())
print((CAMPAIGN / "sweep.yaml").read_text())

## 3. Đăng ký Sweep — thao tác có kết nối Forge

Review config phía trên rồi đặt `REGISTER_SWEEP=True`. Chạy lại cell giữ nguyên ID đã lưu.
Nếu có `registration_started.json` nhưng thiếu `sweep_id.json`, phản hồi mạng có thể đã bị mất:
kiểm dashboard và khôi phục ID thật trước khi thử lại; không tự tạo Sweep thứ hai.

In [ ]:
REGISTER_SWEEP = False
if REGISTER_SWEEP:
    run([sys.executable, "-m", "vn_tsd.pipelines.faster_rcnn.sweep", "register", "--campaign", CAMPAIGN])
    print((CAMPAIGN / "sweep_id.json").read_text())
else:
    print("Chưa đăng ký. Bật REGISTER_SWEEP sau khi review config.")

## 4. Chạy agent — bắt đầu dùng GPU quota

Đặt `START_AGENT=True`, `AGENT_COUNT=1` cho preflight/pilot. Sau khi đã nghiệm thu và chốt budget,
tuning có thể tăng count hữu hạn phù hợp thời lượng session. Không chạy hai agent cùng hai GPU.
Save Version → Save & Run All giúp lưu outputs; bật hai công tắc trước khi Save & Run nếu muốn thực thi.
Mỗi trial fresh COCO weights, không resume checkpoint của trial dở.

Pause/Stop Sweep trong Forge để ngừng nhận trial mới. Cancel/stop notebook là ngắt trial;
giữ artifacts dở, chạy lại bằng run ID mới. Tiếp tục session cần khôi phục campaign về **đúng đường dẫn**,
cùng source SHA/environment/cache, rồi dùng ID đã lưu, không đăng ký Sweep mới.

In [ ]:
START_AGENT = False
AGENT_COUNT = 1
if START_AGENT:
    assert (CAMPAIGN / "sweep_id.json").is_file(), "Đăng ký/khôi phục sweep ID trước."
    run([sys.executable, "-m", "vn_tsd.pipelines.faster_rcnn.sweep", "agent",
         "--campaign", CAMPAIGN, "--count", AGENT_COUNT])
else:
    print("Chưa chạy GPU. Bật START_AGENT khi sẵn sàng.")

## 5. Bàn giao outputs

Mỗi trial có `run.zip`, SHA256, `launch_status.json`, config/metrics/history/checkpoint/log.
Giữ cả failed/interrupted, không chỉ winner. Tải metadata campaign và từng ZIP/checksum bên dưới;
Save Version giữ toàn bộ /kaggle/working. Đừng ZIP toàn repo, dataset, netrc hoặc secrets.
Đưa artifacts local vào `.agents/artifacts/notebooks/<campaign>/` để review.

In [ ]:
import zipfile
from IPython.display import display, FileLink

metadata_zip = CAMPAIGN / "campaign_metadata.zip"
with zipfile.ZipFile(metadata_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(CAMPAIGN.iterdir()):
        if path.is_file() and path.suffix in {".json", ".yaml"}:
            archive.write(path, path.name)
display(FileLink(str(metadata_zip)))
for trial_dir in sorted((CAMPAIGN / "trials").glob("*")):
    status_file = trial_dir / "launch_status.json"
    print(trial_dir.name, json.loads(status_file.read_text()) if status_file.exists() else "chưa có launch receipt")
    for filename in ("run.zip", "run.zip.sha256"):
        if (trial_dir / filename).is_file():
            display(FileLink(str(trial_dir / filename)))

Nguồn SDK/command macros: [CoreWeave Sweeps walkthrough](https://docs.coreweave.com/products/wandb/sweeps/walkthrough),
[Sweep configuration](https://docs.coreweave.com/products/wandb/sweeps/sweep-config-keys).
Local mock/CPU-Gloo tests không thay nghiệm thu Kaggle CUDA/NCCL/Forge thật.